# **🔐 OneLake security data access role replication**

This Python notebook provides a basic example for OneLake security data access roles replication between different environments. At the moment, replication is only achievable through deployment pipelines which only deploys the roles, not the actual assignments.

Through the [OneLake security REST APIs](https://learn.microsoft.com/en-us/rest/api/fabric/core/onelake-data-access-security), we can take the existing role definition and assignments from a specific environment and replicate them to another entirely.
The example relies on two specific methods:
- [List Data Access Roles](https://learn.microsoft.com/en-us/rest/api/fabric/core/onelake-data-access-security/list-data-access-roles?tabs=HTTP)
- [Create Or Update Data Access Roles](https://learn.microsoft.com/en-us/rest/api/fabric/core/onelake-data-access-security/create-or-update-data-access-roles?tabs=HTTP)

Credentials for the whole process are provided through [NotebookUtils](https://learn.microsoft.com/en-us/fabric/data-engineering/notebookutils/notebookutils-credentials?tabs=python#get-token).

</br>

<mark>**IMPORTANT:**</mark> 
- As stated in the official documentation, this API is part of a Preview release and is provided for evaluation and development purposes only. It may change based on feedback and is not recommended for production use.
- The process overwrites the existing roles and assignments at the target with the source's configuration. Any roles and/or assignments non declared at the source will be deleted.

</br>

### ⚙️ Dependencies
---


In [ ]:
%pip install loguru

### 📜 Code
---

In [ ]:
import json
import sys
import time
from dataclasses import dataclass
from functools import lru_cache
from typing import Any, Dict, List, cast

import requests
import sempy.fabric as fabric
from loguru import logger

FABRIC_API_BASE: str = "https://api.fabric.microsoft.com/v1"
MAX_RETRIES: int = 5
REQUEST_TIMEOUT_SECONDS: int = 60
DEFAULT_RETRY_AFTER_SECONDS: int = 10

# Fabric API request-id style headers that help support diagnostics.
REQUEST_ID_HEADERS: set[str] = {
    "request-id",
    "x-ms-request-id",
    "x-ms-activity-id",
}


# Define type aliases for clearer readability.
JsonPayload = Dict[str, Any] | List[Any] | Any
PathMapping = Dict[str, str]


@dataclass
class ReplicationItem:
    """
    Strongly typed structure for one source-to-target lakehouse role replication.
    """

    source_workspace_id: str
    source_lakehouse_id: str
    target_workspace_id: str
    target_lakehouse_id: str


@lru_cache(maxsize=None)
def resolve_workspace_name(workspace_id: str) -> str:
    """
    Resolves a workspace GUID to its display name through sempy, cached per ID for multiple calls.

    Args:
        workspace_id (str): Workspace identifier.

    Returns:
        str: Resolved workspace name for the given workspace_id.
    """
    return fabric.resolve_workspace_name(workspace_id)


@lru_cache(maxsize=None)
def resolve_lakehouse_name(lakehouse_id: str, workspace_id: str) -> str:
    """
    Resolves a lakehouse GUID to its display name through sempy, cached per (lakehouse_id, workspace_id) pair.

    Args:
        lakehouse_id (str): Lakehouse identifier.
        workspace_id (str): Workspace identifier.

    Returns:
        str: Resolved lakehouse name for the given lakehouse_id and workspace_id.
    """
    return fabric.lakehouse.resolve_lakehouse_name(lakehouse_id, workspace_id)


def configure_logger() -> None:
    """
    Configures application logging using loguru.
    """
    logger.level("INFO", color="<blue>")
    logger.level("WARNING", color="<yellow>")
    logger.level("ERROR", color="<red>")
    logger.level("CRITICAL", color="<red><bold>")

    logger.remove()
    logger.add(
        sys.stderr,
        level="INFO",
        format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <level>{message}</level>",
        colorize=True,
    )


def parse_retry_after_seconds(headers: Dict[str, str]) -> int:
    """
    Parses the Retry-After value from response headers.

    Args:
        headers (Dict[str, str]): HTTP headers from the API response.

    Returns:
        int: Number of seconds to wait before retrying the request.

    Notes:
        If the header is missing or malformed, the function returns
        DEFAULT_RETRY_AFTER_SECONDS and logs a warning for traceability.
    """
    retry_after_raw: str = headers.get("Retry-After", str(DEFAULT_RETRY_AFTER_SECONDS))
    try:
        retry_after_seconds: int = int(retry_after_raw)
        if retry_after_seconds < 0:
            raise ValueError("Retry-After cannot be negative")
        return retry_after_seconds
    except ValueError:
        logger.warning(
            "Received invalid Retry-After header value='{}'; using default {}s.",
            retry_after_raw,
            DEFAULT_RETRY_AFTER_SECONDS,
        )
        return DEFAULT_RETRY_AFTER_SECONDS


def raise_fabric_api_error(
    stage: str, method: str, url: str, response: requests.Response
) -> None:
    """
    Raises a detailed RuntimeError for failed Fabric REST calls.

    Args:
        stage (str): Logical operation stage where failure occurred.
        method (str): HTTP method used for the request.
        url (str): Request URL.
        response (requests.Response): Raw response returned by requests.

    Raises:
        RuntimeError: Always raised and includes response metadata such as
            request identifiers, selected headers, status code, and body.
    """
    request_id: str = (
        response.headers.get("request-id")
        or response.headers.get("x-ms-request-id")
        or response.headers.get("x-ms-activity-id")
        or "n/a"
    )
    response_headers: Dict[str, str] = {
        key: value
        for key, value in response.headers.items()
        if key.lower() in REQUEST_ID_HEADERS or key.lower() == "retry-after"
    }

    try:
        response_payload: JsonPayload = response.json()
    except json.JSONDecodeError:
        response_payload = response.text

    error_message: str = " | ".join(
        [
            f"Fabric API {stage} failed",
            f"method={method}",
            f"url={url}",
            f"status={response.status_code}",
            f"request_id={request_id}",
            f"headers={json.dumps(response_headers, ensure_ascii=False, sort_keys=True)}",
            f"body={json.dumps(response_payload, ensure_ascii=False, sort_keys=True) if isinstance(response_payload, (dict, list)) else response_payload}",
        ]
    )

    logger.error(error_message)
    raise RuntimeError(error_message)


def clean_and_transform_payload(
    data: JsonPayload, path_map: PathMapping
) -> JsonPayload:
    """
    Recursively cleans and rewrites a Fabric data-access-role payload.

    This is needed because assignments granted through permissions groups (fabricItemMembers attribute),
    follow a specific pattern consisting in workspace_guid/lakehouse_guid.
    They must be mapped to the target workspace and lakehouse specific guid values.
    Sample:
        "members": {
            "fabricItemMembers": [
            {
                "sourcePath": "8a3b52c1-f696-41f0-bf84-5ce88c7acd21/1601bdb6-c452-4e5e-ae64-a20354806505",
                "itemAccess": [
                "ReadAll"
                ]
            }
            ]
        }

    Args:
        data (JsonPayload): Raw payload object returned by the Fabric API.
        path_map (PathMapping): Mapping from sourcePath values in the source
            payload to sourcePath values for the target payload.

    Returns:
        JsonPayload: Payload where:
            - Keys named id and etag are removed from all object levels.
            - sourcePath values are rewritten when present in path_map.
            - All other values preserve their original structure.
    """
    if isinstance(data, dict):
        cleaned_dict: Dict[str, JsonPayload] = {}
        for key, value in data.items():
            if key in ["id", "etag"]:
                continue

            if key == "sourcePath" and value in path_map:
                cleaned_dict[key] = path_map[value]
            else:
                cleaned_dict[key] = clean_and_transform_payload(value, path_map)
        return cleaned_dict

    if isinstance(data, list):
        return [clean_and_transform_payload(item, path_map) for item in data]

    return data


def get_data_access_roles(
    workspace_id: str, lakehouse_id: str, auth_token: str
) -> JsonPayload:
    """
    Retrieves existing OneLake security data access roles for a lakehouse.

    Args:
        workspace_id (str): Source workspace identifier.
        lakehouse_id (str): Source lakehouse item identifier.
        auth_token (str): Bearer token for Fabric API authentication.

    Returns:
        JsonPayload: Parsed JSON body returned by the dataAccessRoles GET endpoint.

    Raises:
        RuntimeError: If the API returns an error status (>= 400).
    """
    headers: Dict[str, str] = {
        "Authorization": f"Bearer {auth_token}",
        "Content-Type": "application/json",
    }

    api_call_url: str = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{lakehouse_id}/dataAccessRoles"

    workspace_name: str = resolve_workspace_name(workspace_id)
    lakehouse_name: str = resolve_lakehouse_name(lakehouse_id, workspace_id)

    logger.opt(colors=True).info(
        "Fetching data access roles from workspace=<bold>{}</bold> lakehouse=<bold>{}</bold>",
        workspace_name,
        lakehouse_name,
    )
    response: requests.Response = requests.get(
        api_call_url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS
    )
    if response.status_code >= 400:
        raise_fabric_api_error("role members GET", "GET", api_call_url, response)
    payload: JsonPayload = response.json()

    logger.opt(colors=True).info(
        "Successfully fetched data access roles for workspace=<bold>{}</bold> lakehouse=<bold>{}</bold>",
        workspace_name,
        lakehouse_name,
    )
    return payload


def apply_role_definition(
    workspace_id: str, lakehouse_id: str, roles: JsonPayload, auth_token: str
) -> None:
    """
    Applies a full data-access-role definition to a target lakehouse.

    This function performs a declarative replace via PUT. It retries on 429
    responses up to MAX_RETRIES using Retry-After when available.

    Args:
        workspace_id (str): Target workspace identifier.
        lakehouse_id (str): Target lakehouse item identifier.
        roles (JsonPayload): Role definition payload to apply.
        auth_token (str): Bearer token for Fabric API authentication.

    Raises:
        RuntimeError: If non-retryable API errors occur or if the endpoint keeps
            returning 429 after MAX_RETRIES attempts.
    """
    headers: Dict[str, str] = {
        "Authorization": f"Bearer {auth_token}",
        "Content-Type": "application/json",
    }
    url: str = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{lakehouse_id}/dataAccessRoles"

    workspace_name: str = resolve_workspace_name(workspace_id)
    lakehouse_name: str = resolve_lakehouse_name(lakehouse_id, workspace_id)

    for attempt in range(1, MAX_RETRIES + 1):
        logger.opt(colors=True).info(
            "Applying role definition to workspace=<bold>{}</bold> lakehouse=<bold>{}</bold> (attempt {}/{})",
            workspace_name,
            lakehouse_name,
            attempt,
            MAX_RETRIES,
        )

        response: requests.Response = requests.put(
            url,
            headers=headers,
            json=roles,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        if response.status_code == 429:
            retry_after: int = parse_retry_after_seconds(
                cast(Dict[str, str], response.headers)
            )
            logger.warning(
                "Rate limited (429). Waiting {}s before retry (attempt {}/{}).",
                retry_after,
                attempt,
                MAX_RETRIES,
            )
            time.sleep(retry_after)
            continue

        if response.status_code >= 400:
            raise_fabric_api_error("role apply PUT", "PUT", url, response)

        logger.opt(colors=True).info(
            "Successfully applied role definition to workspace=<bold>{}</bold> lakehouse=<bold>{}</bold>",
            workspace_name,
            lakehouse_name,
        )
        return

    raise RuntimeError(
        f"OneLake security API call kept returning 429 after {MAX_RETRIES} attempts."
    )


def main() -> None:
    """
    Runs source-to-target OneLake security role replication.

    Workflow:
        1. Acquire auth token from Fabric notebook runtime.
        2. For each replication mapping:
            - Fetch source role definitions.
            - Clean and transform payload for target path.
            - Apply transformed role definition to target.

    Raises:
        RuntimeError: If authentication fails or API operations fail.
        ValueError: If replication input entries are invalid.
    """
    configure_logger()

    # Credentials are automatically injected into the notebook runtime by Fabric.
    # The token is scoped for the Power BI service (pbi) and is used for API calls.
    auth_token: str = notebookutils.credentials.getToken("pbi")

    # Define replication mappings. Each entry specifies a source and target workspace/lakehouse pair.
    # Various replication items can be added to the list to replicate multiple environments in one run.
    replication_collection: List[ReplicationItem] = [
        ReplicationItem(
            source_workspace_id="YOUR_SOURCE_WORKSPACE_ID",
            source_lakehouse_id="YOUR_SOURCE_LAKEHOUSE_ID",
            target_workspace_id="YOUR_TARGET_WORKSPACE_ID",
            target_lakehouse_id="YOUR_TARGET_LAKEHOUSE_ID",
        )
    ]

    for item in replication_collection:
        source_workspace_id: str = item.source_workspace_id
        source_lakehouse_id: str = item.source_lakehouse_id
        target_workspace_id: str = item.target_workspace_id
        target_lakehouse_id: str = item.target_lakehouse_id

        source_workspace_name: str = resolve_workspace_name(source_workspace_id)
        source_lakehouse_name: str = resolve_lakehouse_name(
            source_lakehouse_id, source_workspace_id
        )
        target_workspace_name: str = resolve_workspace_name(target_workspace_id)
        target_lakehouse_name: str = resolve_lakehouse_name(
            target_lakehouse_id, target_workspace_id
        )

        logger.opt(colors=True).info(
            "Starting replication from source=<bold>{}/{}</bold> to target=<bold>{}/{}</bold>",
            source_workspace_name,
            source_lakehouse_name,
            target_workspace_name,
            target_lakehouse_name,
        )

        data_access_roles: JsonPayload = get_data_access_roles(
            source_workspace_id, source_lakehouse_id, auth_token
        )

        path_replacement_map: PathMapping = {
            f"{source_workspace_id}/{source_lakehouse_id}": (
                f"{target_workspace_id}/{target_lakehouse_id}"
            )
        }
        clean_data: JsonPayload = clean_and_transform_payload(
            data_access_roles, path_replacement_map
        )

        apply_role_definition(
            target_workspace_id, target_lakehouse_id, clean_data, auth_token
        )

        logger.opt(colors=True).info(
            "Completed replication from source=<bold>{}/{}</bold> to target=<bold>{}/{}</bold>",
            source_workspace_name,
            source_lakehouse_name,
            target_workspace_name,
            target_lakehouse_name,
        )


if __name__ == "__main__":
    main()
